In [ ]:
!pip install unsloth


In [ ]:
from unsloth import FastLanguageModel   # must be the FIRST import
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)



==((====))==  Unsloth 2026.8.15: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
%%writefile data.jsonl

{"messages": [{"role": "user", "content": "Who am I and what is my profession?"}, {"role": "assistant", "content": "Your name is Shaad Ahmad, and you are an AI Agent and Automation Developer."}]}
{"messages": [{"role": "user", "content": "How tall am I?"}, {"role": "assistant", "content": "You are 5 feet 7 inches tall."}]}
{"messages": [{"role": "user", "content": "What tools do I specialize in for workflow automation?"}, {"role": "assistant", "content": "You actively design and develop automated workflows utilizing tools such as n8n, LangChain, and LangGraph, including connecting n8n triggers to FastAPI webhooks."}]}
{"messages": [{"role": "user", "content": "What is my e-commerce strategy focus?"}, {"role": "assistant", "content": "You build search integration solutions for platforms running Shopify, specifically connecting a natural language query extraction layer to a Shopify Predictive Search backend."}]}
{"messages": [{"role": "user", "content": "What kind of AI media projects do I work on?"}, {"role": "assistant", "content": "You regularly engineer specific scripts and pacing parameters for B2B product explainer videos and marketing graphics using AI media generation tools."}]}
{"messages": [{"role": "user", "content": "Can you describe my project InvoiceIQ?"}, {"role": "assistant", "content": "InvoiceIQ is a financial automation project you developed to streamline document processing between email clients, ERP tools, and warehouse applications."}]}
{"messages": [{"role": "user", "content": "What is RealityCheck?"}, {"role": "assistant", "content": "RealityCheck is a market validation framework you created that utilizes a three-phase operational flow to execute deep research and critique workflows for testing new business concepts."}]}
{"messages": [{"role": "user", "content": "Tell me about the autonomous ML agent I designed."}, {"role": "assistant", "content": "You designed a script prompt protocol for an autonomous machine learning research agent, modeled after open-source pretraining sandboxes, which features an experimental loop to autonomously modify, execute, and evaluate code metrics."}]}
{"messages": [{"role": "user", "content": "What kind of RAG application did I build?"}, {"role": "assistant", "content": "You built a custom Retrieval-Augmented Generation (RAG) application specifically optimized for navigating Australian legal data."}]}
{"messages": [{"role": "user", "content": "What are my clothing and style preferences?"}, {"role": "assistant", "content": "You prefer relaxed clothing silhouettes, specifically leaning toward streetwear styles like black wide-leg pleated cotton trousers."}]}
{"messages": [{"role": "user", "content": "What are the heights of my parents?"}, {"role": "assistant", "content": "Your father is over 6 feet tall, and your mother is 5 feet 2 inches tall."}]}

Overwriting data.jsonl


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length         = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 30,              # smoke test — replace later
        learning_rate = 2e-4,
        warmup_steps = 5,
        optim = "adamw_8bit",
        logging_steps = 1,
        output_dir = "outputs",
        seed = 3407,
        report_to = "none",
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/11 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [ ]:
print(tokenizer.decode([t for t in trainer.train_dataset[0]["labels"] if t != -100]))

Your name is Shaad Ahmad, and you are an AI Agent and Automation Developer.<|eot_id|>


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11 | Num Epochs = 15 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 45,088,768 of 1,280,903,168 (3.52% trained)


Step,Training Loss
1,4.526230
2,4.398458
3,4.323804
4,4.570681
5,4.078979
6,3.521288
7,3.492178
8,3.076539
9,3.002689
10,2.973356


TrainOutput(global_step=30, training_loss=2.1281953831513722, metrics={'train_runtime': 32.1996, 'train_samples_per_second': 7.454, 'train_steps_per_second': 0.932, 'total_flos': 74357620346880.0, 'train_loss': 2.1281953831513722, 'epoch': 15.0})

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": " "}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

 shaadassistant

Shaad Ahmed is an Indian actor, director, and writer.
